# 05 · Casos legados do código Scilab (v2.2)

O código Scilab original lia ficheiros `caso*.txt`. O conversor `cwnet.io.load_legacy_case`
transforma-os em `DesignCase`; aqui executamos os cinco casos de `legacy/scilab/v2.2/Dados`
e comparamos com os relatórios HTML gerados pelo Scilab (`Relatorios/caso*.htm`).

In [1]:
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from cwnet import load_case, run_design_study
from cwnet.reporting import plots, tables

warnings.filterwarnings("ignore")
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)
CASES = Path("../cases")

In [2]:
from cwnet.io import load_legacy_case
LEGACY = Path("../legacy/scilab/v2.2/Dados")
SCILAB = {  # (total, operating, pump, pipe) kUSD/y from the Scilab reports; the traditional of caso2 is stale (see notebook 02)
    "caso1": {"Traditional (edgar_himmelblau)": (42.815, 23.184, 3.209, 16.422), "Curve 1": (39.811, 20.183, 3.045, 16.583), "Curve 2": (41.768, 24.22, 3.209, 14.339), "Curve 3": (43.939, 26.911, 3.312, 13.716), "Curve 4": (46.292, 29.602, 3.409, 13.281)},
    "caso2": {"Curve 1": (43.746, 18.265, 0, 25.481), "Curve 2": (34.622, 21.918, 3.209, 9.495), "Curve 3": (32.857, 24.353, 3.312, 5.192), "Curve 4": (33.142, 26.789, 3.409, 2.944)},
    "caso3": {"Traditional (edgar_himmelblau)": (46.754, 24.316, 3.409, 19.029), "Curve 1": (47.664, 18.265, 0, 29.399), "Curve 2": (44.418, 21.918, 3.209, 19.291), "Curve 3": (45.76, 24.353, 3.312, 18.095), "Curve 4": (47.27, 26.789, 3.409, 17.072)},
    "caso4": {"Traditional (edgar_himmelblau)": (43.441, 22.744, 3.209, 17.488), "Curve 1": (41.167, 19.940, 3.045, 18.182), "Curve 2": (42.872, 23.927, 3.209, 15.736), "Curve 3": (44.355, 26.587, 3.312, 14.456), "Curve 4": (46.503, 29.246, 3.409, 13.848)},
}
rows = []
for name in ["caso1", "caso2", "caso3", "caso4", "caso5"]:
    case = load_legacy_case(LEGACY / f"{name}.txt", pump_fms=1457.4 / 1123.6)
    study = run_design_study(case)
    evs = [(f"Traditional ({k})", e) for k, e in study.traditional.items()] + [(e.design.pump_curve, e) for e in study.lp]
    for label, ev in evs:
        c = ev.costs
        pub = SCILAB.get(name, {}).get(label, (None,) * 4)
        rows.append({"case": name, "project": case.name, "design": label, "pump": ev.design.pump_curve,
                     "total (kUSD/y)": round(c.total / 1e3, 3), "Scilab": pub[0],
                     "operating": round(c.operating / 1e3, 3), "Scilab ": pub[1],
                     "pump cost": round(c.pump_capital_annualized / 1e3, 3), "Scilab  ": pub[2],
                     "pipe cost": round(c.pipe_capital / 1e3, 3), "Scilab   ": pub[3],
                     "balanced": ev.hydraulically_balanced})
display(pd.DataFrame(rows))

,case,project,design,pump,total (kUSD/y),Scilab,operating,Scilab,pump cost,Scilab,pipe cost,Scilab,balanced
0,caso1,Grassroot design problem - 01,Traditional (edgar_himmelblau),Curve 2,42.813,42.815,23.184,23.184,3.209,3.209,16.420,16.422,False
1,caso1,Grassroot design problem - 01,Curve 1,Curve 1,39.808,39.811,20.183,20.183,3.045,3.045,16.581,16.583,True
2,caso1,Grassroot design problem - 01,Curve 2,Curve 2,41.766,41.768,24.220,24.220,3.209,3.209,14.337,14.339,True
3,caso1,Grassroot design problem - 01,Curve 3,Curve 3,43.936,43.939,26.911,26.911,3.312,3.312,13.714,13.716,True
4,caso1,Grassroot design problem - 01,Curve 4,Curve 4,46.291,46.292,29.602,29.602,3.409,3.409,13.280,13.281,True
5,caso2,Revamp problem - 01,Traditional (edgar_himmelblau),Curve 3,35.888,NaN,22.945,NaN,3.312,NaN,9.632,NaN,False
6,caso2,Revamp problem - 01,Curve 1,Curve 1,43.742,43.746,18.264,18.265,0.000,0.000,25.478,25.481,True
7,caso2,Revamp problem - 01,Curve 2,Curve 2,34.620,34.622,21.917,21.918,3.209,3.209,9.494,9.495,True
8,caso2,Revamp problem - 01,Curve 3,Curve 3,32.856,32.857,24.352,24.353,3.312,3.312,5.192,5.192,True
9,caso2,Revamp problem - 01,Curve 4,Curve 4,33.140,33.142,26.788,26.789,3.409,3.409,2.943,2.944,True


Notas: as linhas *Traditional* dos relatórios Scilab de `caso2` e `caso3` provêm de ficheiros de
resultados antigos (ver notebook 02); aqui o tradicional é recalculado com os dados atuais.
No `caso5` o ficheiro legado atribui `fop = 0.25` a um único cenário; o conversor
normaliza as frações de tempo para 1, pelo que o custo operacional é 4× o do relatório
Scilab. O `caso3` ("rebuild") projeta a rede do revamp como se fosse nova (todas as
tubulações `new`, bomba nova) — Tabela 19 do rascunho do artigo.